In [ ]:
import os, shutil, tempfile
os.environ["HYDRA_FULL_ERROR"]="1"

import warnings
from collections.abc import Sequence
import subprocess

# def performance_tweaks():
#     import torch
#     torch.backends.cudnn.benchmark = True
#     torch.backends.cudnn.deterministic = False
#     torch.use_deterministic_algorithms(False)
#     torch.set_float32_matmul_precision('high')
#     torch.jit.enable_onednn_fusion(True)
#     torch.autograd.set_detect_anomaly(False, False) # type:ignore
#     torch.autograd.profiler.profile(False) # type:ignore
#     torch.autograd.profiler.emit_nvtx(False) # type:ignore

# performance_tweaks() # this has 0 effect btw

PYTHON_BIN = "/home/jj/miniconda3/envs/mapperatorinator/bin/python"
MAPINATOR_DIR = "/var/mnt/issd/files 2/programming/libs/mapperatorinator"
PYTHON_BIN = f"{MAPINATOR_DIR}/Mapperatorinator/.venv/bin/python"

# ---------------------------------------------------------------------------- #
#                                  definitions                                 #
# ---------------------------------------------------------------------------- #

# [
#     "/home/jj/miniconda3/envs/mapperatorinator/bin/python",
#     "inference.py",
#     "-cn",
#     "v30",
#     "audio_path='/var/mnt/ssd/Files/Documents/Музыка/a.mp3'",
#     "output_path='/var/mnt/ssd/Files/Programming/libs/mapperatorinator'",
#     "gamemode=0",
#     "difficulty=5",
#     "year=2023",
#     "hp_drain_rate=5",
#     "circle_size=4",
#     "overall_difficulty=8",
#     "approach_rate=9",
#     "slider_multiplier=1.4",
#     "slider_tick_rate=1",
#     "keycount=4",
#     "cfg_scale=1.0",
#     "temperature=0.9",
#     "top_p=0.9",
#     "export_osz=true",
#     "add_to_beatmap=false",
#     "hitsounded=true",
#     "super_timing=true",
# ]

def _to_string_seq(x):
    if x is None: return ()
    if isinstance(x, str): return (x, )
    return x

def get_command(
    name: str,
    input:str,
    output:str,
    difficulty: float = 5,
    mapper_id: int | str | None = None,
    descriptors: str | Sequence[str] | None = (), # ["jump aim", "clean"]
    negative_descriptors: str | Sequence[str] | None = (), # from discord ["improvisation","tech","bursts"]
    cfg_scale: float | None = None, # 1 # Scale of classifier-free guidance
    temperature: float | None = None, # 0.9
    top_p: float | None = None, # 0.9
    year: int | None = None, # 2023

    super_timing: bool | None = None,

    approach_rate: float | None = None, # 9
    circle_size: float | None = None, # 4
    hp_drain_rate: float | None = None, # 5
    overall_difficulty: float | None = None, # 8
    slider_multiplier: float | None = None, # 1.4,
    slider_tick_rate: float | None = None, # 1

    timer_bpm_threshold: float | None = None, # 0.7
    generate_positions: bool | None = None, # false #  Use diffusion to generate object positions
    refine_iters: int | None = None, # 10, Number of diffusion refinement iterations

    lora_path: str | None = None,
):
    descriptors = _to_string_seq(descriptors)
    negative_descriptors = _to_string_seq(negative_descriptors)

    commands = [
        PYTHON_BIN,
        "inference.py",
        "-cn", "v30", # load v30 config
        "device='auto'",
        # "compile=false",
        f"audio_path='{input}'",
        f"output_path='{output}'",
        "gamemode=0",
        f"difficulty={difficulty}",
        "export_osz=true",
        "add_to_beatmap=false",
        "hitsounded=true",
    ]

    sq = "'"
    if len(descriptors) > 0:
        s = f"'{f'{sq},{sq}'.join(descriptors)}'"
        commands.append(f'descriptors=[{s}]')

    if len(negative_descriptors) > 0:
        s = f"'{f'{sq},{sq}'.join(negative_descriptors)}'"
        commands.append(f'negative_descriptors=[{s}]')

    if lora_path is not None: commands.insert(6, f"lora_path='{lora_path}'")
    if year is not None: commands.append(f"year={year}")
    if mapper_id is not None: commands.append(f"mapper_id={mapper_id}")
    if super_timing is not None: commands.append(f"super_timing={str(bool(super_timing)).lower()}")

    if cfg_scale is not None: commands.append(f"cfg_scale={cfg_scale}")
    if temperature is not None: commands.append(f"temperature={temperature}")
    if top_p is not None: commands.append(f"top_p={top_p}")

    if approach_rate is not None: commands.append(f"approach_rate={approach_rate}")
    if circle_size is not None: commands.append(f"circle_size={circle_size}")
    if hp_drain_rate is not None: commands.append(f"hp_drain_rate={hp_drain_rate}")
    if overall_difficulty is not None: commands.append(f"overall_difficulty={overall_difficulty}")
    if slider_multiplier is not None: commands.append(f"slider_multiplier={slider_multiplier}")
    if slider_tick_rate is not None: commands.append(f"slider_tick_rate={slider_tick_rate}")
    if timer_bpm_threshold is not None: commands.append(f"timer_bpm_threshold={timer_bpm_threshold}")
    if generate_positions is not None: commands.append(f"generate_positions={str(bool(generate_positions)).lower()}")
    if refine_iters is not None: commands.append(f"refine_iters={refine_iters}")

    # determine artist title
    filename = '.'.join(os.path.basename(input).split('.')[:-1])
    if ' - ' in filename:
        artist, title = filename.split(' - ')
    else:
        artist = ''
        title = filename

    commands.append(f"artist='{artist}'")
    commands.append(f"title='{title}'")
    commands.append(f"creator='Mapperatorinator ({name})'")

    return commands


# ---------------------------------------------------------------------------- #
#                                    presets                                   #
# ---------------------------------------------------------------------------- #
presets = []

# ---------------------------------- default --------------------------------- #
presets.append(lambda input,output: get_command(
    input=input,
    output=output,
    name='default',
    year=2023,
    super_timing=True,
    approach_rate=10,
    difficulty=5,
))

# ---------------------------------- Kroytz ---------------------------------- #
presets.append(lambda input,output: get_command(
    input=input,
    output=output,
    name="Kroytz",
    descriptors=["jump aim", "clean"],
    mapper_id=2339768,
    cfg_scale=1.3,
    year=2023,
    super_timing=True,
    approach_rate=10,
    difficulty=5.5,
)) # has jumps on sidewinder second drop, generally good jumps

# ---------------------------------- Sotarks --------------------------------- #
presets.append(lambda input,output: get_command(
    input=input,
    output=output,
    name="Sotarks",
    descriptors=["jump aim", "clean"],
    mapper_id=4452992,
    cfg_scale=1.3,
    year=2023,
    super_timing=True,
    approach_rate=10,
    difficulty=5.5,
)) # good patterns, jumps, no insane parts


# -------------------------------- LoRA-Kroytz-default ------------------------------- #
presets.append(lambda input,output: get_command(
    input=input,
    output=output,
    name="LoRA-Kroytz",
    year=2023,
    super_timing=True,
    approach_rate=10,
    difficulty=5.5,
    lora_path="OliBomby/Mapperatorinator-v30-LoRA-Kroytz",
))

# -------------------------------- LoRA-Kroytz-Kroytz ------------------------------- #
presets.append(lambda input,output: get_command(
    input=input,
    output=output,
    name="LoRA-Kroytz Kroytz",
    descriptors=["jump aim", "clean"],
    mapper_id=2339768,
    cfg_scale=1.3,
    year=2023,
    super_timing=True,
    approach_rate=10,
    difficulty=5.5,
    lora_path="OliBomby/Mapperatorinator-v30-LoRA-Kroytz",
))


# -------------------------------- LoRA-aim-control ------------------------------- #
presets.append(lambda input, output: get_command(
    input=input,
    output=output,
    name="LoRA-aim-control",
    year=2023,
    super_timing=True,
    approach_rate=10,
    difficulty=5.5,
    lora_path="mjoink/Mapperatorinator-v30-LoRA-aim-control",
))

# -------------------------------- LoRA-Arles ------------------------------- #
presets.append(lambda input, output: get_command(
    input=input,
    output=output,
    name="LoRA-Arles",
    year=2023,
    super_timing=True,
    approach_rate=10,
    difficulty=5.5,
    lora_path="mouceen/Mapperatorinator-v30-LoRA-Arles-v1",
))


# ---------------------------------------------------------------------------- #
#                                postprocessing                                #
# ---------------------------------------------------------------------------- #
def packgod(folder, out_dir):
    files = os.listdir(folder)
    osz = ''
    osus = []

    # find osz
    for file in files:
        path = os.path.join(folder, file)
        if path.endswith('.osz'):
            if osz != '': os.remove(path)
            else: osz = path
        elif file.endswith('.osu'):
            osus.append(os.path.join(folder, file))

    # for osu_file in osus:
        # with open(osu_file, 'r+') as f:
        #     osu = f.read()
        #     osu = osu.replace("Version:Mapperatorinator V30", "Version:Mapperatorinator V30\nBeatmapID:0\nBeatmapSetID:-1")


        #     # write
        #     f.seek(0)
        #     f.write(osu)
        #     f.truncate()

    filename = 'unknown'

    # unpack osz
    with tempfile.TemporaryDirectory() as temp:
        shutil.unpack_archive(osz, temp, 'zip')
        for f in os.listdir(temp):
            path = os.path.join(temp, f)
            if path.endswith('.osu'): os.remove(path)
            else: filename = '.'.join(f.split('.')[:-1])


        for osu_file in osus:
            shutil.move(osu_file, temp)

        shutil.make_archive(f"{filename}", 'zip', root_dir=temp)
        shutil.move(f"{filename}.zip", os.path.join(out_dir, f"{filename}.osz"))



# ---------------------------------------------------------------------------- #
#                                      run                                     #
# ---------------------------------------------------------------------------- #
def run(file: str):
    if not os.path.exists(file):
        raise FileNotFoundError(file)
    
    os.chdir(f"{MAPINATOR_DIR}/Mapperatorinator")
    with tempfile.TemporaryDirectory() as tempdir:

        for preset in presets:
            subprocess.run(preset(file, tempdir))

        packgod(tempdir, f"{MAPINATOR_DIR}/out")

### random song suggester AGI

In [2]:
SONGS_DIR = "/var/mnt/issd/files/music/tracks"
import random
os.path.join(SONGS_DIR, random.choice(os.listdir("/var/mnt/issd/files/music/tracks")))

'/var/mnt/issd/files/music/tracks/Liquid Stranger - Run For Cover.mp3'

# Don't forget that script directory is changed!
Use absolute paths only


what I meant is that I use os.chdir so script path is changed, and therefore relative paths become wrong

In [3]:
run("/var/mnt/issd/files/music/tracks/Hudson Lee - Axon.mp3")

Using CUDA for inference (auto-selected).
Using SDPA for attention (auto-selected).
Random seed: 60149
Using default keycount 4
Using default hp_drain_rate 5
Using default circle_size 4
Using default overall_difficulty 8
Using default slider_multiplier 1.4
Using default slider_tick_rate 1
Using default bpm 120
Using default offset 0
Using default source 
Using default preview_time -1
Model loaded: OliBomby/Mapperatorinator-v30 on device cuda
Generating map


100%|██████████| 101/101 [05:55<00:00,  3.52s/it]


Generated beatmap saved to /tmp/tmp4hnoehpg/beatmap9adabbac367d4b99b1c23bdf4e706f32.osu
Generated .osz saved to /tmp/tmp4hnoehpg/beatmapa7abb734d7894e7380e901ee22def1ca.osz
Using CUDA for inference (auto-selected).
Using SDPA for attention (auto-selected).
Random seed: 52851
Using default keycount 4
Using default hp_drain_rate 5
Using default circle_size 4
Using default overall_difficulty 8
Using default slider_multiplier 1.4
Using default slider_tick_rate 1
Using default bpm 120
Using default offset 0
Using default source 
Using default preview_time -1
Model loaded: OliBomby/Mapperatorinator-v30 on device cuda
Generating map


100%|██████████| 101/101 [08:22<00:00,  4.98s/it]


Generated beatmap saved to /tmp/tmp4hnoehpg/beatmap1808fa68058f4775833a4f2952f233fd.osu
Generated .osz saved to /tmp/tmp4hnoehpg/beatmap886d4e337e2b468db1aa3b9a48eb6236.osz
Using CUDA for inference (auto-selected).
Using SDPA for attention (auto-selected).
Random seed: 46383
Using default keycount 4
Using default hp_drain_rate 5
Using default circle_size 4
Using default overall_difficulty 8
Using default slider_multiplier 1.4
Using default slider_tick_rate 1
Using default bpm 120
Using default offset 0
Using default source 
Using default preview_time -1
Model loaded: OliBomby/Mapperatorinator-v30 on device cuda
Generating map


...

100%|██████████| 101/101 [07:39<00:00,  4.55s/it]


Generated beatmap saved to /tmp/tmp4hnoehpg/beatmap298d133866a144ac89cace350b7cc1bb.osu
Generated .osz saved to /tmp/tmp4hnoehpg/beatmap04b4a8b124b24906aedacc7d6376f387.osz
Using CUDA for inference (auto-selected).
Using SDPA for attention (auto-selected).
Random seed: 29149
Using default keycount 4
Using default hp_drain_rate 5
Using default circle_size 4
Using default overall_difficulty 8
Using default slider_multiplier 1.4
Using default slider_tick_rate 1
Using default bpm 120
Using default offset 0
Using default source 
Using default preview_time -1
Loaded LoRA weights from mouceen/Mapperatorinator-v30-LoRA-Arles-v1
Model loaded: OliBomby/Mapperatorinator-v30 on device cuda
Generating map


100%|██████████| 101/101 [08:08<00:00,  4.84s/it]


Generated beatmap saved to /tmp/tmp4hnoehpg/beatmap7ca8abb71fa7436ba2b8afbf896ab9a1.osu
Generated .osz saved to /tmp/tmp4hnoehpg/beatmapdceb09f2a24c427faddc6edaf8064571.osz


FileNotFoundError: [Errno 2] No such file or directory: '/var/mnt/ssd/Files/Programming/libs/mapperatorinator/out/Hudson Lee - Axon.osz'